# GTE embeddings benchmark — French vs. English (STS)

Measures how much embedding quality the English-tuned GTE model loses on French, and
whether the **multilingual** GTE closes that gap. Two models on the same STS-B pairs:

| Model | Served how | Dim |
|-------|-----------|-----|
| `databricks-gte-large-en` | Databricks FMAPI endpoint | 1024 |
| `Alibaba-NLP/gte-multilingual-base` | downloaded from HuggingFace, run **locally on CPU** via `sentence-transformers` | 768 |

- **Dataset:** `stsb_multi_mt` (`en` and `fr` configs) — the same 1,379 STS-B test pairs
  with identical gold similarity scores (0–5). This is MTEB's French STS task.
- **Metric:** Spearman correlation between cosine similarity and gold score
  (`cosine_spearman`, MTEB's standard STS metric), reported per model + language.
- **Compute:** serverless (CPU). **Caching:** dataset and the HF model both cached to a
  UC Volume, so subsequent runs skip the downloads.
- **Charts:** the final cell regenerates the README comparison charts from `results`.

See `SPEC/SPECS.md` for the full specification.

In [ ]:
# mlflow.deployments = FMAPI client; sentence-transformers = local multilingual model;
# matplotlib = charts. transformers is pinned <5 so the model's custom code still finds the
# ModuleUtilsMixin helpers (get_extended_attention_mask, ...) it calls (removed in v5).
# Do NOT pass -U: upgrading numpy breaks serverless's prebuilt pandas/pyspark.
%pip install -q mlflow-skinny matplotlib sentence-transformers hf_transfer einops "transformers>=4.41,<5"
dbutils.library.restartPython()

In [ ]:
# --- Config ---
EN_ENDPOINT = "databricks-gte-large-en"        # Databricks FMAPI embedding endpoint
MULTI_MODEL = "Alibaba-NLP/gte-multilingual-base"  # HuggingFace model, run locally on CPU
HF_DATASET = "PhilipMay/stsb_multi_mt"         # parallel multilingual STS-B (namespaced repo id)
LANGS = ["en", "fr"]
SPLIT = "test"                                 # 1,379 pairs per language
GOLD_MAX = 5.0                                 # STS-B score range is 0..5
FMAPI_BATCH = 100                              # texts per FMAPI request

# UC Volume: dataset cache, HF model cache, results, and chart output.
CATALOG = "lucasbruand_catalog"
SCHEMA = "gte_french_bench"
VOLUME = "data"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
VOLUME_DIR = f"{VOLUME_ROOT}/stsb"
VOL_MODEL = f"{VOLUME_ROOT}/models/gte-multilingual-base"   # persistent HF model cache
RESULTS_CSV = f"{VOLUME_ROOT}/results_gte_french.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
import os
import tempfile
os.makedirs(VOLUME_DIR, exist_ok=True)

# HF_HOME on LOCAL disk: HF's symlink/rename download layout and the trust_remote_code modules
# cache don't work on a FUSE Volume. The model weights are still cached in the Volume (VOL_MODEL)
# via a symlink-dereferenced copy — see the multilingual embedder cell.
os.environ["HF_HOME"] = tempfile.mkdtemp(prefix="hf_local_")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # fast, robust downloads
print(f"Dataset cache: {VOLUME_DIR}\nModel cache:   {VOL_MODEL}\nHF_HOME(local):{os.environ['HF_HOME']}")

In [ ]:
# --- Load STS-B per language, caching to the UC Volume ---
# Download the parquet directly from HuggingFace's parquet API (no `datasets` dependency,
# avoids the serverless /root/.cache permission issue). Download only on cache miss.
import io
import urllib.request
import pandas as pd

def load_sts(lang):
    cache_path = os.path.join(VOLUME_DIR, f"stsb_{lang}_{SPLIT}.parquet")
    if os.path.exists(cache_path):
        print(f"[{lang}] cache hit  -> {cache_path}")
        return pd.read_parquet(cache_path)
    url = f"https://huggingface.co/api/datasets/{HF_DATASET}/parquet/{lang}/{SPLIT}/0.parquet"
    print(f"[{lang}] cache miss -> downloading {url}")
    raw = urllib.request.urlopen(url, timeout=60).read()
    df = pd.read_parquet(io.BytesIO(raw))[["sentence1", "sentence2", "similarity_score"]]
    df.to_parquet(cache_path, index=False)
    print(f"[{lang}] cached {len(df)} rows -> {cache_path}")
    return df

data = {lang: load_sts(lang) for lang in LANGS}
for lang, df in data.items():
    print(f"[{lang}] {df.shape[0]} pairs, score range {df.similarity_score.min()}..{df.similarity_score.max()}")

In [ ]:
# --- Embedder 1: GTE-large-en via FMAPI (MLflow deployments client) ---
import time
import numpy as np
from mlflow.deployments import get_deploy_client

_fmapi = get_deploy_client("databricks")

def _l2norm(arr):
    arr = np.asarray(arr, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return arr / norms

def embed_fmapi(texts, endpoint=EN_ENDPOINT, batch_size=FMAPI_BATCH, max_retries=5):
    """(n, 1024) L2-normalized embeddings from the FMAPI GTE-en endpoint."""
    out = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        for attempt in range(max_retries):
            try:
                resp = _fmapi.predict(endpoint=endpoint, inputs={"input": batch})
                out.extend(item["embedding"] for item in resp["data"])
                break
            except Exception as e:  # rate limit / transient — backoff and retry
                if attempt == max_retries - 1:
                    raise
                wait = 2 ** attempt
                print(f"  batch {start} attempt {attempt+1} failed ({e}); retry in {wait}s")
                time.sleep(wait)
    return _l2norm(out)

print(f"FMAPI embedding dim: {embed_fmapi(['hello world', 'bonjour le monde']).shape[1]}")

In [ ]:
# --- Embedder 2: GTE-multilingual-base from HuggingFace, run locally on CPU ---
# Cache the model in the Volume: HF can't download onto a FUSE Volume, so download to local disk
# then copy in with symlinks dereferenced. Later runs load straight from the Volume, no download.
import torch
import shutil
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer
from transformers.modeling_utils import ModuleUtilsMixin

if os.path.isdir(VOL_MODEL) and os.listdir(VOL_MODEL):
    print(f"model cache HIT  -> {VOL_MODEL}")
else:
    print("model cache MISS -> downloading to local disk, copying to the Volume")
    _snap = snapshot_download(MULTI_MODEL, cache_dir=os.path.join(os.environ["HF_HOME"], "hub"))
    os.makedirs(os.path.dirname(VOL_MODEL), exist_ok=True)
    shutil.copytree(_snap, VOL_MODEL, symlinks=False, dirs_exist_ok=True)

# low_cpu_mem_usage=False forces full CPU init; combined with rebuilding the model's
# non-persistent buffers below, this avoids the uninitialized-buffer IndexError in the RoPE path.
_st = SentenceTransformer(VOL_MODEL, trust_remote_code=True, device="cpu",
                          model_kwargs={"low_cpu_mem_usage": False})
_am = _st[0].auto_model

# Defensive (no-op with transformers<5, which still has these): the custom forward() calls them.
for _m in ("get_extended_attention_mask", "get_head_mask", "invert_attention_mask"):
    if not hasattr(_am, _m):
        setattr(type(_am), _m, getattr(ModuleUtilsMixin, _m))

# Rebuild the NON-persistent buffers (position_ids + RoPE inv_freq/cos/sin) that are absent from
# the checkpoint and can be left uninitialized -> garbage indices -> IndexError in the RoPE branch.
_emb = _am.embeddings
_mpe = int(_am.config.max_position_embeddings)
_emb.position_ids = torch.arange(_mpe)
_re = _emb.rotary_emb
_re.inv_freq = 1.0 / (_re.base ** (torch.arange(0, _re.dim, 2).float() / _re.dim))
_re._set_cos_sin_cache(seq_len=_mpe, device=torch.device("cpu"), dtype=torch.float32)

def embed_local(texts, batch_size=32):
    """(n, 768) L2-normalized embeddings from the local multilingual model (CPU)."""
    emb = _st.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=False)
    return emb.astype(np.float32)

print(f"Local (CPU) embedding dim: {embed_local(['hello world', 'bonjour le monde']).shape[1]}")

In [ ]:
# --- Run the STS benchmark: each model x each language ---
from scipy.stats import spearmanr, pearsonr

EMBEDDERS = {
    "gte-large-en": embed_fmapi,          # FMAPI, English-tuned
    "gte-multilingual-base": embed_local,  # local CPU, multilingual
}

rows = []
for model_name, embed_fn in EMBEDDERS.items():
    for lang in LANGS:
        df = data[lang]
        s1, s2 = df["sentence1"].tolist(), df["sentence2"].tolist()
        gold = df["similarity_score"].to_numpy(dtype=np.float32) / GOLD_MAX
        print(f"[{model_name} | {lang}] embedding {len(s1)} pairs...")

        cos = np.sum(embed_fn(s1) * embed_fn(s2), axis=1)  # inputs already L2-normalized
        spearman = spearmanr(cos, gold).correlation
        pearson = pearsonr(cos, gold)[0]
        rows.append({"model": model_name, "lang": lang, "n_pairs": len(s1),
                     "cosine_spearman": spearman, "cosine_pearson": pearson})
        print(f"[{model_name} | {lang}] cosine_spearman={spearman:.4f}  cosine_pearson={pearson:.4f}")

results = pd.DataFrame(rows).set_index(["model", "lang"])

In [ ]:
# --- Headline: per-model FR/EN ratio + French head-to-head ---
def sp(model, lang):
    return results.loc[(model, lang), "cosine_spearman"]

print("=" * 60)
print(f"Dataset: {HF_DATASET} [{SPLIT}]  (cached in {VOLUME_DIR})")
print("=" * 60)
print(results.round(4).to_string())
print("-" * 60)
for model in EMBEDDERS:
    print(f"{model:24s}  EN={sp(model,'en'):.4f}  FR={sp(model,'fr'):.4f}  FR/EN={sp(model,'fr')/sp(model,'en'):.3f}")
print("-" * 60)
fr_gain = sp("gte-multilingual-base", "fr") - sp("gte-large-en", "fr")
print(f"French Spearman: multilingual vs English-only -> {fr_gain:+.4f}")

results.to_csv(RESULTS_CSV)
print(f"\nSaved: {RESULTS_CSV}")
display(results.reset_index())

In [ ]:
# --- Comparison charts (reproducible from `results`) ---
# Writes both README charts to the UC Volume. To refresh the committed images, copy them
# from VOLUME_ROOT into the repo's gte-french/assets/.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e4e0"
EN_MODEL_C, MULTI_C = "#2a78d6", "#eb6834"   # validated categorical pair (blue / orange)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "font.size": 11,
    "text.color": INK, "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK_2, "font.family": "DejaVu Sans",
})
MODELS = ["gte-large-en", "gte-multilingual-base"]

# Chart 1: Spearman by language, grouped by model
langs_disp = ["English", "French"]
en_model = [sp("gte-large-en", l) for l in LANGS]
multi = [sp("gte-multilingual-base", l) for l in LANGS]
x = np.arange(len(LANGS)); w = 0.34
fig, ax = plt.subplots(figsize=(7.2, 4.4), dpi=150)
for offset, vals, color, label in [(-w/2 - 0.01, en_model, EN_MODEL_C, "gte-large-en (FMAPI)"),
                                    (w/2 + 0.01, multi, MULTI_C, "gte-multilingual-base (CPU)")]:
    bars = ax.bar(x + offset, vals, w, label=label, color=color)
    for r in bars:
        ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.012,
                f"{r.get_height():.3f}", ha="center", va="bottom", fontsize=10, color=INK)
ax.set_ylim(0, 1.0); ax.set_ylabel("cosine_spearman (vs. gold)")
ax.set_xticks(x); ax.set_xticklabels(langs_disp)
ax.set_title("STS-B Spearman: English vs. French, by model",
             fontsize=12.5, color=INK, pad=12, loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, color=GRID, linewidth=1); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper right", fontsize=9.5)
fig.tight_layout(); fig.savefig(f"{VOLUME_ROOT}/sts_by_model.png", facecolor=SURFACE)

# Chart 2: French Spearman head-to-head
fr_scores = [sp(m, "fr") for m in MODELS]
fig2, ax2 = plt.subplots(figsize=(7.2, 3.2), dpi=150)
bars = ax2.bar([0, 1], fr_scores, 0.5, color=[EN_MODEL_C, MULTI_C])
for r, v in zip(bars, fr_scores):
    ax2.text(r.get_x() + r.get_width()/2, v + 0.012, f"{v:.3f}",
             ha="center", va="bottom", fontsize=11, color=INK)
ax2.set_ylim(0, 1.0); ax2.set_ylabel("cosine_spearman (vs. gold)")
ax2.set_xticks([0, 1]); ax2.set_xticklabels(MODELS)
delta = fr_scores[1] - fr_scores[0]
ax2.set_title(f"French STS-B: multilingual vs. English-only  ({delta:+.3f})",
              fontsize=12.5, color=INK, pad=12, loc="left")
ax2.spines[["top", "right"]].set_visible(False)
ax2.yaxis.grid(True, color=GRID, linewidth=1); ax2.set_axisbelow(True)
fig2.tight_layout(); fig2.savefig(f"{VOLUME_ROOT}/french_model_comparison.png", facecolor=SURFACE)

print(f"Charts written to {VOLUME_ROOT}/")
display(fig); display(fig2)